# Knowledge Layer Demo — Agentic RPG Game Master

This notebook demonstrates the knowledge layer built for the project:

1. Loading initial `GameState` from seed JSON
2. Retrieval helpers that build agent inputs from live state + seed data
3. Running each specialist agent (Lore, NPC, Quest) on real context
4. A full orchestrator turn to show routing + placeholder execution still works

**Owner:** Mazin  
**Depends on:** Christopher's `state_manager.py`, `router.py`, `orchestrator.py` and the agent files in `src/agents/`

## Setup

Make sure the notebook kernel is `Python (agentic_rpg_venv)` and the working directory is the project root so `from src...` imports resolve.

In [1]:
import sys, os
from pathlib import Path

# Walk up until we find the project root
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    raise RuntimeError("Could not find project root containing 'src/'")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Load environment variables from .env BEFORE any src imports
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print("Project root:", project_root)
print("CAP6640_API_KEY loaded:", "yes" if os.environ.get("CAP6640_API_KEY") else "no")

Project root: /home/chris/projects/school-notebooks/cap6640-NLP/GroupProject/Agentic_RPG_Game_Master
CAP6640_API_KEY loaded: no


## 1. Load initial game state

`state_manager.build_initial_state()` reads `data/world/world_state.json`, `data/npcs/npc_data.json`, and `data/quests/quest_data.json` and returns a fully populated `GameState`.

In [2]:
from src.state_manager import build_initial_state

state = build_initial_state(
    session_id="demo_knowledge_layer",
    player_action="The player steps into the inn at Oakshade.",
)

print("Location:", state.canonical.location)
print("Current scene:", state.canonical.current_scene)
print("NPCs loaded:", list(state.canonical.npc_states.keys()))
print("Quests loaded:", [q.quest_id for q in state.canonical.active_quests])
print("Turn:", state.meta.turn_id, "| Session:", state.meta.session_id)

Location: Oakshade Village
Current scene: The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.
NPCs loaded: ['mara_innkeeper', 'corvin_stranger', 'harlan_smith']
Quests loaded: ['missing_scout']
Turn: 1 | Session: demo_knowledge_layer


## 2. Retrieval — build agent inputs

The retrieval layer turns `GameState` + seed JSON into ready-to-run agent inputs. This is the bridge between orchestration and the agents.

In [3]:
from src.retrieval import (
    build_lore_agent_input,
    build_npc_agent_input,
    build_quest_agent_input,
)

lore_input = build_lore_agent_input(state)
print("--- Lore Agent Input ---")
print(lore_input.model_dump_json(indent=2))

--- Lore Agent Input ---
{
  "player_action": "The player steps into the inn at Oakshade.",
  "current_scene": "The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.",
  "location": "Oakshade Village",
  "world_summary": "The Shattered Vale is a patchwork of forest-edge villages and ruined holdfasts left behind by a fallen kingdom. Travel between settlements is slow and uncertain, and the Vale's old stone ruins hum with restless things that the living do not fully understand. Oakshade sits at the southern edge of the wood, the last ordered place before the trees grow wild.",
  "relevant_facts": [
    "Several villagers have gone missing near the Emberwood Ruins in the past season.",
    "The Village Council governs Oakshade and is openly worried about the Ashen Band.",
    "Outsiders are tolerated but watched; the innkeeper is the first person most strangers speak to.",
    "The woods between Oakshade and the ruins are considered safe 

In [4]:
npc_input = build_npc_agent_input(state, npc_id="corvin_stranger")
print("--- NPC Agent Input (Corvin) ---")
print(npc_input.model_dump_json(indent=2))

--- NPC Agent Input (Corvin) ---
{
  "player_action": "The player steps into the inn at Oakshade.",
  "npc_name": "Corvin",
  "npc_role": "Hooded traveler at the inn; possible scout or informant with his own agenda",
  "npc_personality": [
    "quiet",
    "guarded",
    "quick to size people up",
    "willing to pay for discreet help"
  ],
  "npc_goals": [
    "Find out what the Ashen Band is pulling out of the Emberwood Ruins",
    "Avoid being identified by the Village Council",
    "Recruit an outsider to do the dangerous part of his investigation"
  ],
  "npc_disposition": "neutral",
  "current_scene": "The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.",
  "relationship_summary": "Has been watching the player since they walked in. Will approach if the player seems capable and discreet.",
  "relevant_memory": [
    "Claims to have followed the Ashen Band for weeks before coming to Oakshade.",
    "Has a partial map of the Ember

In [5]:
quest_input = build_quest_agent_input(state)
print("--- Quest Agent Input ---")
print(quest_input.model_dump_json(indent=2))

--- Quest Agent Input ---
{
  "player_action": "The player steps into the inn at Oakshade.",
  "current_scene": "The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.",
  "active_quest_summaries": [
    "The Missing Scout: not_started"
  ],
  "completed_objectives": [],
  "recent_player_choices": [],
  "world_context": "The player arrives at Oakshade Village, where rumors, danger, and opportunity are beginning to converge.",
  "npc_context": null
}


## 3. Run the Lore Agent

The player looks around the inn and asks about the village. The Lore Agent grounds its response in Oakshade's seeded facts and rumors.

In [6]:
from src.agents.lore_agent import run_lore_agent
from src.state_manager import start_new_turn

state = start_new_turn(state, "I scan the common room and ask what's been going on in Oakshade lately.")
lore_input = build_lore_agent_input(state)

lore_output = await run_lore_agent(lore_input)
print("--- Lore Agent Output ---")
print(lore_output.model_dump_json(indent=2))

--- Lore Agent Output ---
{
  "summary": "The player scans the common room and gathers local knowledge by asking the innkeeper or nearby patrons about recent happenings in Oakshade. Based on the world context, they would pick up on visible social tension, whispered concerns about disappearances, and guarded mentions of the Ashen Band. The innkeeper, as the primary contact for outsiders, is the most likely source of candid (if cautious) information. Villagers may warm to the player slightly if they listen respectfully, but outsiders are watched — overt prying could draw suspicion.",
  "environmental_details": [
    "The common room likely holds a mix of locals — farmers, a trader or two, possibly an off-duty village watchman. Conversations are hushed and faces carry visible worry.",
    "The innkeeper stands behind the bar and is the most approachable figure; they are accustomed to sizing up strangers quickly.",
    "A notice board or spoken word near the bar may reference the Council's

## 4. Run the NPC Agent

The player approaches Corvin, the hooded stranger. The NPC Agent should respond in Corvin's voice — guarded, watchful, and hinting at his own agenda.

In [7]:
from src.agents.npc_agent import run_npc_agent

state = start_new_turn(state, "I walk over to the hooded man in the corner and sit down across from him.")
npc_input = build_npc_agent_input(state, npc_id="corvin_stranger")

npc_output = await run_npc_agent(npc_input)
print("--- NPC Agent Output (Corvin) ---")
print(npc_output.model_dump_json(indent=2))

--- NPC Agent Output (Corvin) ---
{
  "speech": "Corvin doesn't look up immediately. He lets a few seconds pass, turning his cup slowly on the table before his eyes lift to meet yours — calm, measuring. \"Bold move, sitting down uninvited.\" A pause. \"Either you're confident, or you haven't learned yet that confidence and caution aren't the same thing.\" He glances once toward the bar — toward Mara — then back to you, his voice dropping just slightly. \"You came in from the south road. Alone. No guild mark, no militia colors.\" It isn't a question. He leans back just slightly, arms loose, but his posture never fully relaxes. \"What brings someone like you to a village like Oakshade? And think before you answer. I'm not asking to make conversation.\"",
  "emotional_tone": "Guarded, calculating, quietly testing — surface-level neutral with an undercurrent of focused interest.",
  "intent": "Corvin is vetting the player before risking any exposure. He spotted the player when they entered

### NPC comparison — the same action at Mara

Running the same turn against a different NPC should produce a clearly different voice. This is a quick sanity check that personality/role/goals actually flow through.

In [8]:
state = start_new_turn(state, "I lean on the bar and ask the innkeeper what's been happening around here.")
npc_input_mara = build_npc_agent_input(state, npc_id="mara_innkeeper")

npc_output_mara = await run_npc_agent(npc_input_mara)
print("--- NPC Agent Output (Mara) ---")
print(npc_output_mara.model_dump_json(indent=2))

--- NPC Agent Output (Mara) ---
{
  "speech": "*Mara sets down the clay mug she's been polishing and gives you a long, measured look before speaking.*\n\n\"Happening around here.\" *She repeats it like she's tasting the question.* \"Depends on what kind of 'happening' you're after, friend. We get traders through. Farmers. The occasional wanderer looking for a warm bed and a quiet night.\" *She leans her forearms on the bar, voice dropping just slightly.* \"Most of 'em find both and move on by morning.\"\n\n*She glances briefly toward the far corner of the common room — just a flicker — before her eyes come back to you.*\n\n\"I'll be honest with you the way I'm honest with everyone who walks through that door: Oakshade is a peaceful village. We intend to keep it that way. If you're here to drink and rest, you're welcome as long as your coin's good and you keep your business to yourself.\" *A pause, deliberate.* \"If you're here for something else... well. I'd want to know a little more 

## 5. Run the Quest Agent

The player accepts Mara's request to look into the missing scout. The Quest Agent should interpret this as advancing the `speak_to_mara` objective.

In [9]:
from src.agents.quest_agent import run_quest_agent

state = start_new_turn(state, "I tell Mara I'll look into what happened to the missing scout.")
quest_input = build_quest_agent_input(state)

quest_output = await run_quest_agent(quest_input)
print("--- Quest Agent Output ---")
print(quest_output.model_dump_json(indent=2))

--- Quest Agent Output ---
{
  "progress_interpretation": "The player's action of agreeing to investigate the missing scout directly advances \"The Missing Scout\" quest from not_started to in_progress. By explicitly committing to Mara, the player has accepted the quest hook and established Mara as a key NPC contact. This is a clean, unambiguous progression trigger — no stall, failure, or conflict detected.",
  "objective_status_updates": [
    "The Missing Scout: Update status from not_started to in_progress.",
    "Mark first objective as active: 'Speak to Mara and agree to investigate the missing scout' — COMPLETED by this action.",
    "Unlock next objective: 'Gather information about the scout's last known whereabouts or mission.'"
  ],
  "new_branches_or_hooks": [
    "Mara Hook — Mara may have additional details (the scout's name, last patrol route, or what they were searching for) that can be extracted in follow-up dialogue, potentially branching the investigation toward the fo

## 6. Full orchestrator turn (routing + placeholder execution)

Finally, confirm that Christopher's orchestrator still runs end-to-end with the new seed data. The agents aren't wired in yet — nodes execute as placeholders — but routing, state transitions, and event logging all flow through.

In [10]:
from src.orchestrator import run_turn

result = run_turn(state, player_action="I ask Mara who else I should talk to in the village.")

print("Selected route:", result.selected_route)
print("Next nodes:   ", result.next_nodes)
print("Execution plan:", result.execution_plan)
print("Aborted:      ", result.aborted)
print("Reason:       ", result.reason)
print()
print("Execution results:")
for r in result.execution_results:
    print(f"  - {r.node_name}: {r.status} ({r.attempts} attempt(s)) — {r.summary}")

Selected route: dialogue
Next nodes:    ['npc_agent']
Execution plan: ['npc_agent']
Aborted:       False
Reason:        Player action appears to be directed at a character or conversation.

Execution results:
  - npc_agent: completed (1 attempt(s)) — Placeholder execution completed for npc_agent.


In [11]:
print("--- Turn events ---")
for e in result.state.turn.current_turn_events:
    print(f"[{e.event_type}] ({e.source_node}) {e.summary}")

--- Turn events ---
[route_selected] (orchestrator) Route 'dialogue' selected with nodes ['npc_agent'].
[route_reason] (router) Player action appears to be directed at a character or conversation.
[execution_plan_built] (orchestrator) Execution plan created: ['npc_agent']
[node_executed] (orchestrator) npc_agent executed with placeholder behavior.


## What this proves

- Seed JSON loads cleanly into a validated `GameState`.
- Retrieval packages that state into the correct input shape for all three agents.
- Each agent runs against real seeded context and returns structured output.
- Different NPCs produce clearly different voices from the same kind of prompt.
- Christopher's orchestrator still routes and executes turns correctly with the new data.

## What's still placeholder

- The orchestrator executes nodes as placeholders rather than calling the real agents.
- Wiring `run_lore_agent`, `run_npc_agent`, and `run_quest_agent` into `execute_planned_nodes` is the next orchestration-side step — ideally coordinated with Christopher.